# 00 — Kaggle environment setup (run first, no model weights)

Validates GPU + logs the Sec 13 env header + HF auth. No weights downloaded here — loaders in `02` do that.

In [ ]:
# Cell 0 — fetch repo (branch Sayok) on Kaggle; safe to re-run
import os, subprocess
if os.path.isdir("/kaggle/working"):
    dest = "/kaggle/working/Testing-dip"
    if os.path.isdir(os.path.join(dest, ".git")):
        subprocess.run(["git", "-C", dest, "pull", "--ff-only"], check=False)
    elif not os.path.isdir(dest):
        subprocess.run(["git", "clone", "-b", "Sayok",
                        "https://github.com/iamahnaf/Testing-dip.git", dest], check=True)
    print("repo:", dest)
else:
    print("not on Kaggle — skipping repo fetch (run from the repo checkout)")


In [ ]:
# Cell 1 — system deps (poppler for pdf2image; tectonic optional, PIL fallback exists)
!apt-get update -qq && apt-get install -y -qq poppler-utils > /dev/null 2>&1
!which tectonic || echo "tectonic missing -> 01 will use PIL fallback"

In [ ]:
# Cell 2 — python deps (Qwen3.5 needs transformers from main)
!pip install -q "transformers[serving] @ git+https://github.com/huggingface/transformers.git@main" accelerate bitsandbytes torchao "qwen-vl-utils>=0.0.14" pillow pdf2image pandas matplotlib rouge-score huggingface_hub

In [ ]:
# Cell 3 — Sec 13 env header (paste this into every result log)
import sys, platform, torch, transformers
from datetime import datetime, timezone
print("timestamp:", datetime.now(timezone.utc).isoformat())
print("python:", sys.version.split()[0], "| platform:", platform.platform())
print("torch:", torch.__version__, "| cuda avail:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv 2>/dev/null || echo "no nvidia-smi"
!nvcc --version 2>/dev/null | tail -1 || echo "no nvcc"
print("洛塔 T4 note: Turing sm75 -> use fp16 + sdpa, NOT bf16/flash-attn-2")

In [ ]:
# Cell 4 — HF auth via Kaggle Secrets (HF_TOKEN). Skips gracefully if absent (public models still work).
import os
from huggingface_hub import login
tok = os.environ.get("HF_TOKEN", "")
try:
    from kaggle_secrets import UserSecretsClient
    tok = tok or UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("kaggle_secrets unavailable:", e)
if tok:
    login(token=tok)
    print("HF login OK")
else:
    print("No HF_TOKEN — continuing anonymously (gated models will fail in 02)")

In [ ]:
# Cell 5 — resolve model IDs (documents Sec 3 substitution if names drift)
from huggingface_hub import model_info
PRIMARY = "Qwen/Qwen3.5-9B"   # biggest Qwen3.5 that fits T4 16GB in 4-bit
try:
    mi = model_info(PRIMARY)
    print(f"OK  {PRIMARY}  sha={mi.sha[:12]} pipeline={mi.pipeline_tag}")
except Exception as e:
    print(f"MISS {PRIMARY}: {e}")
print("\nNo fallback model — if PRIMARY is missing/renamed, stop and record it.")
